In [ ]:
import datasets
import json
from pathlib import Path

In [ ]:
subreddit = "technepal"
posts_path = Path.home() / f"Downloads/r_{subreddit}_posts.jsonl"
comments_path = Path.home() / f"Downloads/r_{subreddit}_comments.jsonl"

In [ ]:
! tail -n 3 $comments_path

In [ ]:
import duckdb
query = f"""
    select 
        name as id, author, title, selftext, ups, downs, num_comments,
        created_utc,
    from read_json(
    '{posts_path}',
        format = 'newline_delimited',
        columns = {{
            name: 'VARCHAR',
            author: 'VARCHAR',
            title: 'VARCHAR',
            selftext: 'VARCHAR',
            ups: 'INT64',
            downs: 'INT64',
            num_comments: 'INT64',
            created_utc: 'INT64',
            quarantine: 'BOOL'
        }}
    )
    where 1=1
    and (title <> '[deleted by user]' and selftext <> '[removed]')
    and length(title)  > 5 and length(selftext)  > 5
    and coalesce(quarantine, false) = false
"""
posts_df = duckdb.query(query).df()
posts_df

In [ ]:
from datasets import Features, Value

posts_features = Features({
    "id": Value("string"),
    "author": Value("string"),
    "title": Value("string"),
    "selftext": Value("string"),
    "ups": Value("int64"),
    "downs": Value("int64"),
    "num_comments": Value("int64"),
    "created_utc": Value("int64"),
})

posts_ds = datasets.Dataset.from_pandas(posts_df, features=posts_features, preserve_index=False)
posts_ds.features

In [ ]:
query = f"""
    select
        name as id, link_id as post_id, parent_id, author,
        ups, downs, body, created_utc
    from read_json(
    '{comments_path}',
        format = 'newline_delimited',
        columns = {{
            name: 'VARCHAR',
            link_id: 'VARCHAR',
            parent_id: 'VARCHAR',
            author: 'VARCHAR',
            ups: 'INT64',
            downs: 'INT64',
            body: 'VARCHAR',
            created_utc: 'INT64'
        }}
    )
"""
comments_df = duckdb.query(query).df()
comments_df

In [ ]:
comments_features = Features({
    "id": Value("string"),
    "post_id": Value("string"),
    "parent_id": Value("string"),
    "author": Value("string"),
    "ups": Value("int64"),
    "downs": Value("int64"),
    "body": Value("string"),
    "created_utc": Value("int64"),
})

comments_ds = datasets.Dataset.from_pandas(comments_df, features=comments_features, preserve_index=False)
comments_ds.features

In [ ]:
repo_id = "jangedoo/nepali-reddit"

posts_ds.push_to_hub(
    repo_id,
    config_name='posts',
    split=subreddit,
    commit_message=f"Add posts for r/{subreddit}"
)

comments_ds.push_to_hub(
    repo_id,
    config_name='comments',
    split=subreddit,
    commit_message=f"Add comments for r/{subreddit}"
)